# Forecast


In [ ]:
from fastapi import APIRouter, Depends
from sqlalchemy.ext.asyncio import AsyncSession
from sqlalchemy import select, func
from typing import Optional
from app.database import get_db
from app.models.user import User, Transaction
from app.schemas.schemas import ForecastResult
from app.services.auth_service import get_current_user
from app.services.ml_service import get_forecast_service

router = APIRouter(prefix="/forecast", tags=["forecast"])


In [ ]:
@router.post("/spending", response_model=ForecastResult)
async def forecast_spending(
    days: int = 7,
    user: User = Depends(get_current_user),
    db: AsyncSession = Depends(get_db),
):
    """
    Forecast daily spending for the next N days based on user's transaction history.
    """
    svc = get_forecast_service()

    # Pull last 90 days of expense history
    result = await db.execute(
        select(Transaction)
        .where(
            Transaction.user_id == user.id,
            Transaction.transaction_type == "expense",
        )
        .order_by(Transaction.transaction_date.desc())
        .limit(90)
    )
    txs = result.scalars().all()

    # Build daily series
    import pandas as pd
    daily_totals = {}
    for t in txs:
        date = t.transaction_date.date().isoformat()
        daily_totals[date] = daily_totals.get(date, 0) + t.amount

    # Pad to 30 days minimum
    history = list(daily_totals.values())
    history.reverse()
    if len(history) < 30:
        history = [0.0] * (30 - len(history)) + history

    forecast = svc.predict(history[-90:])

    return ForecastResult(
        predicted_amount=forecast["predicted_daily_avg"] * days,
        period="daily",
        confidence=forecast["confidence"],
    )
